In [1]:
# load the convnext delta

import sys
from pathlib import Path

# Notebooks live in notebooks/; project packages are at the repo root.
sys.path.insert(0, str(Path.cwd().resolve().parent))

from models.backbones.delta_convnext import DeltaConvNext
import torch

model = DeltaConvNext()
# rewire() replaces stage3 with sharedBlock + delta blocks, which is the layout the checkpoint stores.
model.rewire()
state_dict = torch.load("../outputs/outputs/delta_convnextv1_imagenet/weights/last.pth", map_location="cpu")
model.load_state_dict(state_dict["model"])

Rewired: stage3 -> 1 shared block + 9 deltas + 2 tail layers


<All keys matched successfully>

In [2]:
# Print attributes of model.deltifiedStage3[0] block
print([k for (k,v) in model.deltifiedStage3[0].deltas()])

['dwWDelta', 'lnWDelta', 'pw1WDelta', 'pw2WDelta', 'lsGDelta', 'dwBDelta', 'lnBDelta', 'pw1BDelta', 'pw2BDelta']


In [3]:
mean_deltas = {}
total_blocks = 9
for i in range(9):
    for key, delta in model.deltifiedStage3[i].deltas():
        acc = mean_deltas.get(key, torch.zeros_like(delta)) + delta * 1 / total_blocks
        mean_deltas[key] = acc

In [ ]:
# Compute ratios =  mean(delta)^2 / mean(delta^2)
ratios = {}
squared_deltas = {}
squared_mean = 
for i in range(9):
    squared_deltas[i] = {}
    for key, delta in model.deltifiedStage3[i].deltas():
        squared_deltas[i][key] = torch.sum(delta**2)





In [5]:
# Set mean deltas to all blocks
for i in range(9):
    model.deltifiedStage3[i].setDeltas(mean_deltas)

In [4]:
# Check that we are using mean deltas
breakFor = False
for i in range(9):
    for key, delta in model.deltifiedStage3[i].deltas():
        expected = mean_deltas[key].to(delta.device)
        if not torch.equal(delta, expected):
            print(f"Block {i}, delta {key} does not match the mean delta")
            print(f"difference norm: {torch.norm(delta - expected)}")
            breakFor = True
            break
    if breakFor:
        break


Block 0, delta dwWDelta does not match the mean delta
difference norm: 10.650224685668945


In [5]:
# Validate
import os
from pathlib import Path
from torch.utils.data import DataLoader
from data.imagenet import ImageNetDataset
from data.transforms.transforms import build_train_batch_transforms, build_val_transforms
from engine.validator import Validator
from metrics.metricHistory import DictHistoryMetrics
from metrics.metricLoss import MetricLoss
from metrics.top1acc import Top1AccMetric
from models.backbones.delta_convnext import DeltaConvNext
from timm.loss import SoftTargetCrossEntropy
import torch

BATCH_SIZE = 128
OUTPUT_PATH = Path("../outputs/delta_convnext_validation")

def available_cpus() -> int:
    """Cores this process may use (SLURM cgroup aware)."""
    slurm_cpus = os.environ.get("SLURM_CPUS_PER_TASK")
    if slurm_cpus:
        return int(slurm_cpus)
    return len(os.sched_getaffinity(0))


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)
model.setUseDeltas(True)

# No DataParallel here: the delta blocks keep the shared block as an unregistered attribute,
# so the replicas on the other GPUs would still read tensors that live on cuda:0.
print(f"Using {device}")

# train loader and val loader of imagenet with huggingface datasets

val_dataset = ImageNetDataset(split="validation", transforms=build_val_transforms())

num_workers = int(available_cpus() // 2)
print(f"Dataloader workers: {num_workers}")

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=True,
    persistent_workers=True,
)

validator = Validator(
    model=model,
    criterion=SoftTargetCrossEntropy(),
    device=device,
    val_loader=val_loader,
    batch_transforms=build_train_batch_transforms(),
    num_classes=1000,
    amp=False,
)
val_histoy_metrics = DictHistoryMetrics(OUTPUT_PATH, split="val")
val_histoy_metrics.addHistoryMetric("top1acc", Top1AccMetric)
val_histoy_metrics.addHistoryMetric("loss", MetricLoss, higher_is_better=False)
validator.validate(val_histoy_metrics)


Using cuda
Dataloader workers: 10


100%|██████████| 391/391 [01:47<00:00,  3.63it/s, loss=1.03] 

val_top1acc: 0.74582
val_loss: 1.132816184654236


### Base model performance:

val_top1acc: 0.74582

val_loss: 1.132816184654236

### Setting mean delta as deltas:

val_top1acc: 0.28026

val_loss: 3.814050767364502

### Removing deltas:

val_top1acc: 0.23012

val_loss: 4.0967326908874515